# Fine tuning an Italian GPT on the 'Divine Comedy'

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch 
import urllib.request
import re 
import torch.nn.functional as F
import math 
import pandas as pd

torch.manual_seed(42)


url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
with urllib.request.urlopen(url) as response:
    dante_text = response.read().decode('utf-8')

model_name = "LorenzoDeMattei/GePpeTto"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print(model)

[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 29999), got 50256. This may result in unexpected behavior.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(30000, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=30000, bias=False)
)


### Data cleaning

In [2]:
# Canto headers, e.g. "Inferno: Canto I", "Purgatorio: Canto XXXIII"
canto_header_re = re.compile(r'^(Inferno|Purgatorio|Paradiso):\s*Canto\s+[IVXLCDM]+\s*$')

# Front-matter title lines
title_lines = {
    "LA DIVINA COMMEDIA",
    "di Dante Alighieri",
    "INFERNO",
    "PURGATORIO",
    "PARADISO",
}

def clean_editorial_lines(text):
    cleaned = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in title_lines:
            continue
        if canto_header_re.match(stripped):
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)

dante_text = clean_editorial_lines(dante_text)
print(len(dante_text)) 

534889


In [3]:
# sanity check
text = "Nel mezzo del cammin di nostra vita"
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)


print(tokenizer)
print(tokens)
print(ids)

GPT2Tokenizer(name_or_path='LorenzoDeMattei/GePpeTto', vocab_size=30000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
['Nel', 'Ġmezzo', 'Ġdel', 'Ġcamm', 'in', 'Ġdi', 'Ġnostra', 'Ġvita']
[1771, 2312, 280, 4881, 266, 272, 1930, 991]


### One forward pass, by hand

Manually walking logits -> softmax -> argmax -> top-k, before relying on any HF convenience methods.

In [4]:
prompt = "Nel mezzo del cammin di nostra vita"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print(inputs["input_ids"].shape)
print(outputs.logits.shape)  # (batch, sequence, vocab)


torch.Size([1, 8])
torch.Size([1, 8, 30000])


In [5]:
# Prediction for the next token
next_token_logits = outputs.logits[:, -1, :]
probs = F.softmax(next_token_logits, dim=-1)
print(probs.sum())  # sanity check: should be 1.0

next_token_id = torch.argmax(probs, dim=-1)
print(tokenizer.decode(next_token_id))

top_probs, top_ids = torch.topk(probs, 10)
for prob, token_id in zip(top_probs[0], top_ids[0]):
    print(f"{tokenizer.decode(token_id)!r}: {prob.item():.4f}")

tensor(1.0000)
,
',': 0.3417
' c': 0.0516
'.': 0.0515
' ci': 0.0433
' e': 0.0323
' non': 0.0254
' si': 0.0240
' è': 0.0152
' il': 0.0129
' vi': 0.0122


### Greedy decoding demo

prompt -> next token -> append -> predict again -> append -> ...
This loop is the basic mechanism behind text generation, before using `model.generate()`.

In [6]:
generated = inputs["input_ids"]  # this part initially used argmax (picking the most probable next token-often repeating sentences)

for _ in range(50):
    with torch.no_grad():
        outputs = model(generated)

    next_token_logits = outputs.logits[:, -1, :]

    # Convert logits into probabilities
    probabilities = torch.softmax(next_token_logits, dim=-1)

    # Sample the next token
    next_token_id = torch.multinomial(probabilities, num_samples=1)

    generated = torch.cat([generated, next_token_id], dim=1)

print(tokenizer.decode(generated[0]))

Nel mezzo del cammin di nostra vita non dimenticateci che lo Stato romano romano anni prima era stato un nemico delle nostre regioni. L'Imperatrice longobarda assediٍ quindi Napoli, e costrinse a venire con una mossa per aggredirla. San Gregorio XII scioglieva


---
# Fine tuning the model


In [7]:
dante_ids = tokenizer.encode(dante_text)

# Data spltting same as trasformer

data = torch.tensor(dante_ids, dtype=torch.long)

n1 = int(0.9 * len(data))
n2 = int(0.95 * len(data))

train = data[:n1]
val = data[n1:n2]
test = data[n2:]

### 4 - Batching

In [8]:
def get_batch(split, batch_size, block_size):
    if split == 'train':
        data_split = train
    elif split == 'val':
        data_split = val
    elif split == 'test':
        data_split = test
    else:
        raise ValueError('Split must be "train", "val" or "test"')

    ix = torch.randint(0, len(data_split) - block_size, (batch_size,))

    X = torch.stack([
        data_split[index : index + block_size] for index in ix
    ])  # input tokens

    Y = torch.stack([
        data_split[index + 1 : index + block_size + 1] for index in ix
    ])  # same sequence shifted one token to the right

    return X, Y

### 5 - Hyperparameters

In [9]:
batch_size = 4
block_size = 128
learning_rate = 5e-5 # Model already pretrained
max_steps = 100
eval_interval = 10


xb, yb = get_batch('train', batch_size, block_size)
print(xb.shape, yb.shape)
print(tokenizer.decode(xb[0]))

torch.Size([4, 128]) torch.Size([4, 128])
 cortese portinaio:
"Venite dunque a' nostri gradi innanzi".
  Là ne venimmo; e lo scaglion primaio
bianco marmo era sì pulito e terso,
ch'io mi specchiai in esso qual io paio.
  Era il secondo tinto più che perso,
d'una petrina ruvida e arsiccia,
crepata per lo lungo e per traverso.
  Lo terzo, che di sopra s'ammassiccia,
porfido mi parea, sì fiammeggiante,
come sangue che fuor di vena spiccia.


### 6 - Device placement
This is done so that the notebook can run on Colab.

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = model.to(device)

cpu


### 7 - Baseline loss check

Loss on the *untouched* pretrained model, before any gradient steps.



In [11]:
xb, yb = get_batch('train', batch_size, block_size)
xb, yb = xb.to(device), yb.to(device)

with torch.no_grad():
    outputs = model(input_ids=xb, labels=xb)

loss = outputs.loss.item()
print(f"Baseline loss: {loss:.4f}")
print(f"Baseline perplexity: {math.exp(loss):.2f}")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline loss: 6.0235
Baseline perplexity: 413.04


### 8 - Evaluation function

In [12]:
@torch.no_grad()
def estimate_loss(splits, eval_iters=10):
    model.eval()
    losses = {}

    for split in splits:
        # 1. Quick random batch estimate for training loop progress
        if split == 'train':
            split_losses = []
            for _ in range(eval_iters):
                xb, yb = get_batch(split, batch_size, block_size)
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(input_ids=xb, labels=xb)
                split_losses.append(outputs.loss.item())
            losses[split] = sum(split_losses) / len(split_losses)

        # 2. Full deterministic pass in non-overlapping 128-token windows for val and test
        else:
            data_split = val if split == 'val' else test
            split_losses = []

            for i in range(0, len(data_split) - block_size + 1, block_size):
                chunk = data_split[i : i + block_size].unsqueeze(0).to(device)
                outputs = model(input_ids=chunk, labels=chunk)
                split_losses.append(outputs.loss.item())

            losses[split] = (
                sum(split_losses) / len(split_losses) if split_losses else 0.0
            )

    model.train()
    return losses

In [13]:
# Zero-shot baseline evaluation on non-overlapping windows
base_losses = estimate_loss(['val', 'test'])
print(f"Zero-shot Val Loss:  {base_losses['val']:.4f} | Perplexity: {math.exp(base_losses['val']):.2f}")
print(f"Zero-shot Test Loss: {base_losses['test']:.4f} | Perplexity: {math.exp(base_losses['test']):.2f}")

Zero-shot Val Loss:  6.3006 | Perplexity: 544.91
Zero-shot Test Loss: 6.3146 | Perplexity: 552.59


### 13 - Optimizer + training loop

Optimizer is created fresh here, right before the loop starts — no demo cell has touched
the model or optimizer state beforehand (see section 10).

In [14]:
torch.manual_seed(42)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_steps):
    model.train()

    xb, yb = get_batch('train', batch_size, block_size)
    xb, yb = xb.to(device), yb.to(device)

    outputs = model(input_ids=xb, labels=xb)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss(['train', 'val'], eval_iters=10)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

step 0: train loss 5.1812, val loss 5.2351
step 10: train loss 4.4800, val loss 4.5150
step 20: train loss 4.3939, val loss 4.3933
step 30: train loss 4.1743, val loss 4.3255
step 40: train loss 4.1248, val loss 4.2864
step 50: train loss 4.2317, val loss 4.2575
step 60: train loss 4.1223, val loss 4.2240
step 70: train loss 4.0490, val loss 4.2218
step 80: train loss 4.0284, val loss 4.2033
step 90: train loss 4.0790, val loss 4.1720


### 14 - Saving model + tokenizer

In [15]:
save_path = "./gepPetto-dante"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gepPetto-dante/tokenizer_config.json', './gepPetto-dante/tokenizer.json')

### 15 - Qualitative sampling

In [16]:
def generate_text(prompt, max_new_tokens=150, temperature=0.8, top_k=50):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

prompts = [
    "Nel mezzo del cammin di nostra vita",
    "Nel mezzo",
    "Amor che ne la mente mi ragiona",
]

for p in prompts:
    print(generate_text(p))
    print("---")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo del cammin di nostra vita, ma quivi non tien l'occhio,
che non ti farai alcun segno.
  Non vedi, non vedi, né vedi
che è meglio non voler che tu ti sia.
  E tu che non ti dia in questa la via
e non vedi, ma ti senti che ti dia
sanza piaghe per quella che vedi.
  Però non fia che non m'appaioni.
  Così come non mi vedi, che non mi vedi, che mi ti basti
e mi chiedi che non ti sia mai, che seppi la bocca,
e che mi rivolsi a te, e da' suoi dolci
pur dimandarmi.
  E se tu li occhi miei
---


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo di quella notte,
la pioggia d'amare e di fare,
s'infiammava per li altri, e non per il suo fiato.
  Ma l'altro, ch'era e ch'a sua voce era sùbito,
s'adocchiava in alto a quel ch'era prima;
e per suo buon consiglio non s'attendeva.
  Ed elli a me: "Non mi disfare",
che son contento ch'ella s'aiutasse e tu seppi,
e che l'avea fatto a quel che 'l viso facea.
  Allor che già m'el se lagri
inetto
di sé, mi disse: "
---
Amor che ne la mente mi ragiona
del peccato, che non mi piace".
  Amor mi rispuose: "Maestro mio, perché mi duole?".
  E io: "Lei mi dice: "Tuo Dio mi rida".
  Ed elli: "Io, che ti dir non ti credo
ma come mi diedi?".
  Ma lui: "Chi è costui con la licenza?".
  Lo guardar si mise a vedere: "Maestro, tu hai un Dio?".
  Li occhi e la faccia e la faccia, e la testa e le gambe si mossero,
  "Da te la mia mente non ti sentir ti dir".
  Poi 'l gridò: "Perché mi dici 'l
---


In [17]:
### 16 - Final test metrics

full_ft_losses = estimate_loss(['val', 'test'])

print("Full fine-tune test loss:", full_ft_losses['test'])
print("Full fine-tune test perplexity:", math.exp(full_ft_losses['test']))

Full fine-tune test loss: 4.189741269866032
Full fine-tune test perplexity: 66.0057110844944


---
### LoRA test

In [18]:
# --- Cell A: fresh base model + LoRA wrapping ---
from peft import LoraConfig, get_peft_model, TaskType
torch.manual_seed(42)

# Reload the pretrained weights so LoRA starts from the untouched base model
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # tells PEFT this is a next-token model
    r=8,                            # rank of the update: W' = W + (alpha/r) * B @ A,
                                    # with A: (r, in) and B: (out, r)
    lora_alpha=16,                  # scaling factor; the update is multiplied by alpha/r
    lora_dropout=0.05,              # dropout on the LoRA branch input (regularisation)
    target_modules=["c_attn"],      # fused Q/K/V projection in each GPT2Attention block
    fan_in_fan_out=True,            # GPT-2 uses Conv1D (weights stored transposed
                                    # vs nn.Linear); without this the update is misapplied
)

# Freezes every base weight and injects trainable A and B matrices into c_attn
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # check against your hand calculation (~0.3M)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 109,177,344 || trainable%: 0.2701


In [19]:
# --- Cell B: sanity check ---
# B is initialised to zeros, so the LoRA model must equal the base model at step 0.
# This loss should match your corrected zero-shot baseline.
print(estimate_loss(['train', 'val'], eval_iters=20))

{'train': 6.172187280654907, 'val': 6.300628541120842}


In [20]:
# --- Cell C: hyperparameters (overwrite the earlier ones) ---
batch_size = 8
block_size = 128
learning_rate = 2e-4    # LoRA tolerates (and needs) a higher LR than full fine-tuning
max_steps = 1000        # print len(train) to work out how many epochs this is
eval_interval = 100

In [21]:
# --- Cell D: optimizer + training loop ---
# Only pass parameters with requires_grad=True: the LoRA A/B matrices.
# The frozen base weights have no gradients and no optimizer state.
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate,
)

for step in range(max_steps):
    model.train()
    xb, yb = get_batch('train', batch_size, block_size)
    xb, yb = xb.to(device), yb.to(device)

    loss = model(input_ids=xb, labels=xb).loss   # HF shifts internally

    optimizer.zero_grad()
    loss.backward()          # gradients flow only into the LoRA matrices
    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss(['train', 'val'], eval_iters=20)
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")

step 0: train 6.1083, val 6.2577
step 100: train 4.4606, val 4.5099
step 200: train 4.3070, val 4.3553
step 300: train 4.2527, val 4.2834
step 400: train 4.1883, val 4.2504
step 500: train 4.1591, val 4.2137
step 600: train 4.1174, val 4.1838
step 700: train 4.1125, val 4.1710
step 800: train 4.1228, val 4.1490
step 900: train 4.0788, val 4.1468


In [22]:
# --- Cell E: save adapter only (~1 MB, not the full model) ---
model.save_pretrained("./geppetto-dante-lora")

# To reload later:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_name)
# model = PeftModel.from_pretrained(base, "./geppetto-dante-lora")

In [23]:
# --- Cell F: qualitative sampling with the LoRA model ---
# Redefines generate_text so it works with the PEFT-wrapped model.
# It uses the global `model` (now the LoRA model), `tokenizer` and `device`.
def generate_text(prompt, max_new_tokens=150, temperature=0.8, top_k=50):
    model.eval()                                   # disable dropout for generation
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,                   # keyword arg: PEFT wrappers prefer this
            max_new_tokens=max_new_tokens,         # how many tokens to add after the prompt
            do_sample=True,                        # sample instead of greedy (greedy loops)
            temperature=temperature,               # <1 = safer/more predictable, >1 = wilder
            top_k=top_k,                           # sample only from the 50 likeliest tokens
            pad_token_id=tokenizer.eos_token_id,   # silences the padding warning
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

prompts = [
    "Nel mezzo del cammin di nostra vita",
    "Nel mezzo",
    "Amor che ne la mente mi ragiona",
]

for p in prompts:
    print(generate_text(p))
    print("---")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo del cammin di nostra vita l'aurora.
  Non mi piace quel poco che io soglian le stelle,
per ch'io non udii, ma che 'ntelletto
né da me, né da lui mai vidi,
  né da' miei occhi furon tanto più invecchi,
che 'l poeta mi disse: "Se tu ti ami
che la tua lingua
tieni, sì che tu non vedi, e se tu mi senti!",
  e questa volta non mi disse: "Se tu te ne lamenti,
che tu, come che 'ntelletto tieni,
non vedi, io non se' perdirai né con ciٍ che tu mi per te, né per
---


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo di tutte le cose fumi.

  Ma come, dove sì ch'io fia, ch'io veggio
d'un'uno di più che di più che di più;
però che 'l mio canto s'ignea, se non ch'i' lesse;
  "non ch'al più al più non son men che li occhi",
sì che 'l suo canto m'è sempre più,
e 'l mio canto che già m'è stato sùbito;
  e la mia bocca, non per ciel, s'accostò a lui,
s'andò a lui come in quella che 'l mondo non gliene.
  Io
---
Amor che ne la mente mi ragiona
che la testa, come d'un piè del tutto la memoria,
e quella, che non mi pare, che li occhi più savi.
  E io che son stato io a fare la terra,
e che 'nna veggo che 'l mondo è stato,
con la morte a colui ch'è morto;
  perché è suo, se non lo è stato, il nome che si vede
più che altro a coloro che mi son venuti.
  E io che son morto, a voi ch'io non mi son venuto,
non per me, che non ho creduto, e non mi sono creduto,
per le buone parole che vi ho dette o per me,
  che io
---


In [24]:
# --- Cell G: Final LoRA test metrics ---
final_losses = estimate_loss(['val', 'test'])
test_perplexity = math.exp(final_losses['test'])

print("LoRA Test loss:", final_losses['test'])
print("LoRA Test perplexity:", test_perplexity)

LoRA Test loss: 4.125042822823596
LoRA Test perplexity: 61.87045866137689
